# 🛡️ Federated Learning Training Pipeline (FL-IoT-IDS)
## 🚀 Decentralized Edge Intrusion Detection with Prebuilt Modules

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/federated_training_pipeline.ipynb)

This notebook provides an automated, publication-grade **Federated Learning simulation pipeline** for IoT Network Intrusion Detection using the **CICIoT2023** benchmark. It is powered entirely by the prebuilt, decoupled Python modules in `src/`:

* **Data Engineering & Preprocessing (`src.data`):** Leak-free preprocessing (`TabularDataPreprocessor`), `RobustScaler`, $\log(1+x)$ skew compression, and inverse-frequency class weights.
* **Edge Statistical Heterogeneity (`src.data.partitioner`):** Stratified IID (`StratifiedIIDPartitioner`) and Non-IID Dirichlet skew simulation (`DirichletNonIIDPartitioner`, $\alpha \in \{0.5, 0.1\}$).
* **Edge-Optimized Architecture (`src.models`):** Ultra-lightweight MLP backbone (`TabularIoTMLPModel`, ~54.28 KB, 13.9K params) free of BatchNorm drift.
* **Class Imbalance Loss (`src.losses`):** `MultiClassFocalLoss` suppressing majority DDoS/DoS gradients while amplifying minority intrusions (`Web-based`, `Brute-force`).
* **Edge Client Optimization (`src.training`):** `FederatedClientTrainer` with FedProx proximal regularization ($\mu$) and gradient clipping.
* **Server Coordination & Aggregation (`src.federated`):** Turnkey server strategies (`FedAvgStrategy`, `FedProxStrategy`, `FedMedianStrategy`, `FedTrimmedMeanStrategy`).
* **Publication Visualization (`src.visualization`):** Turnkey 300-DPI diagnostic plots (client distributions, round convergence, confusion matrices, and scenario comparisons).

---
### 🛠️ Step 1: Environment Setup & Hardware Acceleration
We check GPU acceleration, synchronize the latest codebase from GitHub, and install all required dependencies.

In [ ]:
# 1. Hardware acceleration verification
import torch
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device:   {device}")
if torch.cuda.is_available():
    print(f"GPU Model:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Clone or pull the latest repository codebase
import os, sys, subprocess

repo_url = "https://github.com/NVKQ2022/FedLearning.git"
repo_dir = "/content/FedLearning"

if not os.path.exists(repo_dir):
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
else:
    print("Repository already present. Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", "origin/main"], check=True)

os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Purge any cached in-memory src modules so fresh code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

print(f"✅ Working directory set to: {os.getcwd()}")


In [ ]:
# 3. Install dependencies from requirements.txt
!pip install -q --no-warn-conflicts -r requirements.txt
print("✅ Dependencies verified and installed.")


---
### ⚙️ Step 2: Global Hyperparameter Configuration & Reproducibility Seed
**Edit any hyperparameters here before running the simulation.** All subsequent cells consume settings directly from this single source of truth container (`ExperimentConfig`).

In [ ]:
from src.utils.config import ExperimentConfig
from src.utils.seed import set_seed

# ==============================================================================
# 🛠️ EDIT HYPERPARAMETERS HERE (Single Source of Truth)
# ==============================================================================
CONFIG = ExperimentConfig(
    experiment_name="federated_e5_fedprox_noniid",
    seed=42,                            # Integer seed for 100% bit-level reproducibility
    
    # --- Dataset & Sampling ---
    sample_size=100000,                 # 100,000 for rapid execution, or None for full 5.11M rows
    test_size=0.2,                      # 20% holdout test set (unseen by all clients)
    val_size=0.1,                       # 10% centralized server validation set
    scaler_type="robust",               # 'robust' (median/IQR) or 'standard'
    batch_size=64,                      # Client local batch size
    
    # --- Model Architecture ---
    hidden_dims=(128, 64),              # Lightweight edge funnel (13.9K params, 54.28 KB)
    dropout_rate=0.2,                   # Edge regularization
    
    # --- Loss & Imbalance Strategy ---
    loss_type="focal_loss",             # 'focal_loss', 'cross_entropy', or 'weighted_ce'
    class_weight_strategy="balanced",   # 'balanced' (inverse frequency), 'sqrt_balanced', or 'none'
    focal_gamma=2.0,                    # Focusing parameter (downweights easy background traffic)
    
    # --- Optimization Hyperparameters ---
    optimizer_type="adamw",             # 'adamw', 'adam', or 'sgd'
    learning_rate=1e-3,                 # Client learning rate
    weight_decay=1e-4,                  # Decoupled L2 regularization
    max_grad_norm=5.0,                  # Client gradient clipping
    
    # --- Federated Learning Parameters ---
    num_clients=5,                      # Number of participating IoT edge clients (K)
    num_rounds=10,                      # Server communication rounds (T)
    local_epochs=2,                     # Client local training epochs per round (E)
    federated_strategy="fedprox",       # 'fedprox', 'fedavg', 'fedmedian', 'fedtrimmedmean'
    mu=0.05,                            # FedProx proximal coefficient (mu=0.0 reduces to FedAvg)
    partition_type="dirichlet",         # 'dirichlet' (heterogeneous) or 'iid' (uniform)
    dirichlet_alpha=0.5                 # Dirichlet concentration (0.1: severe skew, 0.5: moderate)
)

# Enforce global deterministic seed
set_seed(CONFIG.seed)

# Display formatted configuration
CONFIG.display()


---
### 💾 Step 3: Google Drive Mounting & Dataset Acquisition
We mount Google Drive to load the real **991.00 MB** dataset (`merged_CICIOT2023_data.csv`).
If the CSV is not found, an automatic synthetic generator will create a compatible dataset so the pipeline can dry-run immediately.

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil
import pandas as pd, numpy as np

target_csv_path = CONFIG.target_csv_path
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

gdrive_csv = CONFIG.gdrive_csv
if not os.path.exists(gdrive_csv):
    if os.path.exists(CONFIG.alt_gdrive_csv):
        gdrive_csv = CONFIG.alt_gdrive_csv

# 2. Acquire real dataset or fallback to synthetic dry-run
if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f"Copying real 991 MB dataset from Google Drive to local SSD ({gdrive_size:.2f} MB)...")
        shutil.copy(gdrive_csv, target_csv_path)
    print(f"✅ REAL CICIoT2023 dataset ready on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
elif os.path.exists(target_csv_path) and os.path.getsize(target_csv_path) >= 100 * 1024 * 1024:
    print(f"✅ REAL CICIoT2023 dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
else:
    print(f"⚠️ Real dataset NOT found at '{gdrive_csv}'.")
    print("Generating synthetic CICIoT2023-compatible dataset (21 MB) for immediate dry-run...")
    n_samples = 30000
    classes = ['Benign', 'DDoS', 'DoS', 'Recon', 'Spoof', 'Mirai', 'Web-based', 'Brute-force']
    p = [0.18, 0.38, 0.23, 0.10, 0.05, 0.04, 0.015, 0.005]
    
    mock_data = {f'feat_{i}': np.random.exponential(scale=1.5, size=n_samples) for i in range(39)}
    mock_data['feat_0'] = np.random.pareto(a=1.5, size=n_samples) * 10.0
    mock_data['group_class_name'] = np.random.choice(classes, size=n_samples, p=p)
    
    pd.DataFrame(mock_data).to_csv(target_csv_path, index=False)
    print(f"⚠️ Synthetic dataset created ({os.path.getsize(target_csv_path)/(1024**2):.2f} MB).")


---
### 🔄 Step 4: Leak-Free Preprocessing & Class Weights
We use `load_and_preprocess_ciciot2023` to perform:
* **Stratified train/val/test split (70/10/20)** strictly before computing statistics.
* **Outlier & skew mitigation:** `RobustScaler` (median & IQR) and $\log(1+x)$ compression.
* **Class weight calculation:** Configurable strategy to penalize minority attack misclassifications.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
import numpy as np

print(f"Loading and preprocessing CICIoT2023 (sample_size={CONFIG.sample_size}, scaler={CONFIG.scaler_type})...")
data = load_and_preprocess_ciciot2023(
    csv_path=target_csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(
    y=data["y_train"],
    num_classes=data["num_classes"],
    strategy=CONFIG.class_weight_strategy
)
class_names = data["class_names"]
minority_classes = ["Web-based", "Brute-force"]

print("\n--- Dataset Summary ---")
print(f"Global Training set:   {len(data['X_train']):,} samples")
print(f"Server Validation set: {len(data['X_val']):,} samples")
print(f"Holdout Test set:      {len(data['X_test']):,} samples")
print(f"Input Features:        {data['input_dim']}")
print(f"Detected Classes ({data['num_classes']}): {class_names}")
print(f"Class Weights ({CONFIG.class_weight_strategy}): {np.round(class_weights, 3)}")


---
### 📊 Step 5: Edge Client Partitioning & Statistical Skew Visualization
We partition the centralized training dataset across simulated IoT edge clients:
* **IID Partitioning (`StratifiedIIDPartitioner`):** Uniform, identical class distribution across all clients (Scenario E2).
* **Dirichlet Non-IID Partitioning (`DirichletNonIIDPartitioner`):** Real-world statistical heterogeneity where rare attacks are concentrated in specific network nodes (Scenario E4/E5).
* **Turnkey Visualization:** We plot the client class distribution using `plot_client_distribution` from `src.visualization`.

In [ ]:
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    get_client_distribution,
    create_client_dataloaders
)
from src.eda import plot_partition_heterogeneity
from src.visualization import plot_client_distribution

# 1. Select Partitioning Scheme
if CONFIG.partition_type.lower() == "iid":
    print(f"Partitioning into {CONFIG.num_clients} Stratified IID clients...")
    partitioner = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed)
    client_partitions = partitioner.partition(data["y_train"])
    dist_title = f"Scenario E2: Stratified IID Partitioning ({CONFIG.num_clients} Clients)"
else:
    print(f"Partitioning into {CONFIG.num_clients} Non-IID clients (Dirichlet alpha={CONFIG.dirichlet_alpha})...")
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=100,
        seed=CONFIG.seed
    )
    client_partitions = partitioner.partition(data["y_train"])
    dist_title = f"Non-IID Dirichlet Partitioning (alpha={CONFIG.dirichlet_alpha}, {CONFIG.num_clients} Clients)"

# 2. Summary Contingency Table with normalized Shannon entropy via src.eda
dist_df = get_client_distribution(client_partitions, data["y_train"], class_names=class_names)
print("
--- Client Partition Heterogeneity Profile ---")
print(dist_df[["client_id", "total_samples", "dominant_class", "dominant_pct", "entropy"]].to_string(index=False))

# 3. Publication-Grade Client Distribution & Heterogeneity Plots
os.makedirs("reports/figures", exist_ok=True)
fig_dist = plot_client_distribution(
    distribution=dist_df,
    title=dist_title,
    save_path="reports/figures/federated_client_distribution.png",
    show=True
)
fig_het = plot_partition_heterogeneity(
    summary_df=dist_df,
    class_names=class_names,
    title=dist_title,
    save_path="reports/figures/federated_client_heterogeneity.png",
    show=True
)

# 4. Build PyTorch DataLoaders for each client
client_loaders = create_client_dataloaders(
    X=data["X_train"],
    y=data["y_train"],
    client_partitions=client_partitions,
    batch_size=CONFIG.batch_size,
    shuffle=True
)
print(f"✅ Instantiated {len(client_loaders)} local edge DataLoaders (batch_size={CONFIG.batch_size}).")


---
### 🧠 Step 6: Neural Network Architecture & Server Aggregation Strategy
We instantiate:
1. **`TabularIoTMLPModel`:** Edge-friendly, BatchNorm-free MLP to prevent running statistic drift across non-IID clients.
2. **`MultiClassFocalLoss`:** Focal loss with $\alpha$-class weights to counter extreme class imbalance.
3. **Server Strategy (`src.federated.strategies`):** Built via `build_strategy` (`FedAvg`, `FedProx`, `FedMedian`, or `FedTrimmedMean`).

In [ ]:
from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.strategies import build_strategy

# 1. Initialize Global Model
global_model = TabularIoTMLPModel(
    input_dim=data["input_dim"],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data["num_classes"],
    dropout_rate=CONFIG.dropout_rate
).to(device)

# 2. Build Loss Function
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights,
    gamma=CONFIG.focal_gamma,
    device=device
)

# 3. Instantiate Federated Aggregation Strategy
strategy = build_strategy(
    strategy_name=CONFIG.federated_strategy,
    mu=CONFIG.mu,
    fraction_fit=1.0,
    min_fit_clients=CONFIG.num_clients
)

print(global_model.summary())
print(f"Active Server Strategy: {strategy.name} (mu={CONFIG.mu if hasattr(strategy, 'mu') else 0.0})")


---
### 🔄 Step 7: Federated Multi-Round Simulation Engine
We execute $T = CONFIG.num_rounds$ communication rounds:
1. **Broadcast:** Global parameters $w_t$ sent to each local client.
2. **Local Training:** `FederatedClientTrainer` trains for $E = CONFIG.local_epochs$ with proximal constraint $\mu$.
3. **Server Aggregation:** `strategy.aggregate_fit()` combines client model updates using sample-weighted averaging.
4. **Global Evaluation:** Server validates current global weights against unseen validation set `val_loader` and checkpoints the top round.

In [ ]:
import copy
from src.optimizers.optimizer import build_optimizer
from src.training.federated_trainer import FederatedClientTrainer
from src.evaluation.evaluator import evaluate_comprehensive

print(f"Starting Federated Simulation: {CONFIG.num_clients} clients, {CONFIG.num_rounds} rounds, {CONFIG.local_epochs} local epochs...")
print(f"Strategy: {strategy.name} | Loss: {CONFIG.loss_type} | Proximal mu: {CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0}")
print("=" * 75)

round_history = {
    "round": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": []
}

best_val_macro_f1 = -1.0
best_round = 0
best_global_weights = None
current_mu = CONFIG.mu if CONFIG.federated_strategy.lower() == "fedprox" else 0.0

for round_idx in range(1, CONFIG.num_rounds + 1):
    client_results = []
    
    # --- 1. Client Local Training Phase ---
    for client_id in range(CONFIG.num_clients):
        # Clone global model for local training
        local_model = copy.deepcopy(global_model)
        local_optimizer = build_optimizer(
            model=local_model,
            optimizer_type=CONFIG.optimizer_type,
            lr=CONFIG.learning_rate,
            weight_decay=CONFIG.weight_decay
        )
        client_trainer = FederatedClientTrainer(
            model=local_model,
            optimizer=local_optimizer,
            criterion=criterion,
            device=device,
            max_grad_norm=CONFIG.max_grad_norm
        )
        
        # Train local client
        loss_c, acc_c = client_trainer.train_epochs(
            dataloader=client_loaders[client_id],
            num_epochs=CONFIG.local_epochs,
            global_model=global_model,
            mu=current_mu
        )
        
        num_samples_c = len(client_partitions[client_id])
        client_results.append((
            local_model.get_weights(),
            num_samples_c,
            {"loss": loss_c, "acc": acc_c}
        ))
    
    # --- 2. Server Aggregation Phase ---
    aggregated_weights, agg_metrics = strategy.aggregate_fit(round_idx, client_results)
    global_model.set_weights(aggregated_weights)
    
    # --- 3. Server Global Validation Phase ---
    val_metrics = evaluate_comprehensive(
        model=global_model,
        dataloader=data["val_loader"],
        class_names=class_names,
        criterion=criterion,
        device=device
    )
    
    t_loss = agg_metrics.get("train_loss", 0.0)
    t_acc = agg_metrics.get("train_acc", 0.0)
    v_loss = val_metrics["loss"]
    v_acc = val_metrics["accuracy"]
    v_f1 = val_metrics["macro_f1"]
    
    round_history["round"].append(round_idx)
    round_history["train_loss"].append(t_loss)
    round_history["train_acc"].append(t_acc)
    round_history["val_loss"].append(v_loss)
    round_history["val_acc"].append(v_acc)
    round_history["val_macro_f1"].append(v_f1)
    
    is_best = v_f1 > best_val_macro_f1
    if is_best:
        best_val_macro_f1 = v_f1
        best_round = round_idx
        best_global_weights = copy.deepcopy(global_model.get_weights())
    
    star_mark = "⭐ (Best)" if is_best else ""
    print(f"[Round {round_idx:02d}/{CONFIG.num_rounds:02d}] Train Loss: {t_loss:.4f} - Train Acc: {t_acc*100:5.2f}% | Val Loss: {v_loss:.4f} - Val Acc: {v_acc*100:5.2f}% - Val Macro-F1: {v_f1*100:5.2f}% {star_mark}")

print("=" * 75)
print(f"🏆 Best Round: {best_round:02d} with Validation Macro-F1: {best_val_macro_f1*100:.2f}%")
if best_global_weights is not None:
    global_model.set_weights(best_global_weights)
    print("✅ Restored global model to best validation checkpoint.")


---
### 📈 Step 8: Federated Convergence Diagnostics
We visualize multi-round convergence trajectories using `plot_federated_convergence` from `src.visualization`.

In [ ]:
from src.visualization import plot_federated_convergence

# 1. Plot Validation Macro-F1 Trajectory across rounds
fig_f1 = plot_federated_convergence(
    rounds_data={f"{strategy.name} (Val Macro-F1)": round_history["val_macro_f1"]},
    metric_name="Macro-F1",
    title=f"Federated Convergence Trajectory ({CONFIG.experiment_name})",
    save_path="reports/figures/federated_convergence_macro_f1.png",
    show=True
)

# 2. Plot Validation Loss Trajectory across rounds
fig_loss = plot_federated_convergence(
    rounds_data={f"{strategy.name} (Val Loss)": round_history["val_loss"]},
    metric_name="Validation Loss",
    title=f"Federated Loss Convergence ({CONFIG.experiment_name})",
    save_path="reports/figures/federated_convergence_loss.png",
    show=True
)


---
### 🏆 Step 9: Comprehensive Holdout Test Evaluation
We evaluate the best global model on the 20% holdout test set (unseen by any client during training):
* **Overall Accuracy & Macro-F1 Score**
* **Per-Class Breakdown (Precision, Recall, F1)**
* **Minority Class Recall Isolation (`Web-based`, `Brute-force`)**
* **Publication Figures:** Confusion matrix heatmap, per-class bar chart, and minority recall.

In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix, plot_per_class_metrics, plot_minority_recall

test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=data["test_loader"],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print("=" * 65)
print(f"🏆 FEDERATED BENCHMARK FINAL RESULTS ({CONFIG.experiment_name})")
print("=" * 65)
print(f"Strategy:         {strategy.name} (mu={CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0})")
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print("-" * 65)
print("🎯 Minority Class Recall (Stealth Penetration Attacks):")
for cls, rec in test_results["minority_recall"].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print("-" * 65)
print("📊 Per-Class Breakdown:")
for cname in class_names:
    f1 = test_results["per_class_f1"][cname] * 100
    rec = test_results["per_class_recall"][cname] * 100
    prec = test_results["per_class_precision"][cname] * 100
    supp = test_results["support"][cname]
    print(f"   {cname:<12} | F1: {f1:5.1f}% | Recall: {rec:5.1f}% | Prec: {prec:5.1f}% | Support: {supp:,}")
print("=" * 65)

# --- Generate Publication Diagnostic Figures ---
# 1. Row-Normalized Confusion Matrix Heatmap
fig_cm = plot_confusion_matrix(
    cm=test_results["confusion_matrix"],
    class_names=class_names,
    title=f"Federated ({strategy.name}): Confusion Matrix",
    save_path="reports/figures/federated_confusion_matrix.png",
    show=True
)

# 2. Per-Class Precision, Recall, and F1 Breakdown
fig_metrics = plot_per_class_metrics(
    metrics_dict=test_results,
    class_names=class_names,
    title=f"Federated ({strategy.name}): Per-Class Performance Breakdown",
    save_path="reports/figures/federated_per_class_metrics.png",
    show=True
)

# 3. Minority Attack Recall Isolation
fig_minority = plot_minority_recall(
    minority_data=test_results["minority_recall"],
    title=f"Federated ({strategy.name}): Minority Attack Recall Isolation",
    save_path="reports/figures/federated_minority_recall.png",
    show=True
)


---
### 📊 Step 10: Multi-Scenario Benchmark Comparison
We compare this Federated Learning run against the Centralized Upper Bound (Scenario E1) using `plot_scenario_comparison`.

In [ ]:
from src.visualization import plot_scenario_comparison
import json

# Load Centralized E1 result if available
centralized_metrics_file = "reports/centralized_e1_metrics.json"
centralized_f1 = 0.884  # Default benchmark baseline reference
if os.path.exists(centralized_metrics_file):
    with open(centralized_metrics_file) as f:
        c_metrics = json.load(f)
        centralized_f1 = c_metrics.get("macro_f1", centralized_f1)

comparison_dict = {
    "E1: Centralized\n(Upper Bound)": centralized_f1,
    f"{strategy.name}\n(Current Run)": test_results["macro_f1"]
}

fig_comp = plot_scenario_comparison(
    results=comparison_dict,
    metric="macro_f1",
    title="Benchmark Comparison: Centralized vs Federated",
    save_path="reports/figures/federated_scenario_comparison.png",
    show=True
)


---
### 💾 Step 11: Export Checkpoints, Config & Reports to Google Drive
We serialize the best global model weights (`.pth`), preprocessor state (`.pkl`), active hyperparameter config (`.json`), test metrics (`.json`), and round history (`.json`), then permanently archive them and all generated figures directly to `/content/drive/MyDrive/NguyenVietKyQuanKLTN/`.

In [ ]:
import json

# 1. Save artifacts locally
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)

model_save_path = f"checkpoints/{CONFIG.experiment_name}_best_model.pth"
global_model.save_weights(model_save_path)

preprocessor_save_path = "checkpoints/preprocessor.pkl"
data["preprocessor"].save(preprocessor_save_path)

config_save_path = f"checkpoints/{CONFIG.experiment_name}_config.json"
CONFIG.save_json(config_save_path)

metrics_export = {k: v for k, v in test_results.items() if k != "confusion_matrix"}
metrics_file = f"reports/{CONFIG.experiment_name}_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(metrics_export, f, indent=4)

history_file = f"reports/{CONFIG.experiment_name}_round_history.json"
with open(history_file, "w") as f:
    json.dump(round_history, f, indent=4)

print(f"✅ Best model checkpoint saved to: {model_save_path}")
print(f"✅ Preprocessor saved to:         {preprocessor_save_path}")
print(f"✅ Config saved to:               {config_save_path}")
print(f"✅ Metrics report saved:         {metrics_file}")
print(f"✅ Round history saved:          {history_file}")

# 2. Archive all artifacts and figures to Google Drive
gdrive_dest = CONFIG.checkpoints_dir
gdrive_reports = CONFIG.reports_dir

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dest, exist_ok=True)
    os.makedirs(gdrive_reports, exist_ok=True)
    
    shutil.copy(model_save_path, os.path.join(gdrive_dest, os.path.basename(model_save_path)))
    shutil.copy(preprocessor_save_path, os.path.join(gdrive_dest, "preprocessor.pkl"))
    shutil.copy(config_save_path, os.path.join(gdrive_dest, os.path.basename(config_save_path)))
    shutil.copy(metrics_file, os.path.join(gdrive_reports, os.path.basename(metrics_file)))
    shutil.copy(history_file, os.path.join(gdrive_reports, os.path.basename(history_file)))
    
    figures_to_copy = [
        "federated_client_distribution.png",
        "federated_convergence_macro_f1.png",
        "federated_convergence_loss.png",
        "federated_confusion_matrix.png",
        "federated_per_class_metrics.png",
        "federated_minority_recall.png",
        "federated_scenario_comparison.png"
    ]
    for fig_file in figures_to_copy:
        src_fig = os.path.join("reports/figures", fig_file)
        if os.path.exists(src_fig):
            shutil.copy(src_fig, os.path.join(gdrive_reports, fig_file))
            
    print(f"\n🎉 All model checkpoints, configs, history & 300-DPI figures permanently archived to Google Drive: {gdrive_dest}")
else:
    print("Google Drive not mounted. Results saved locally in Colab session.")
